In [2]:
import requests
from collections import defaultdict

# URL dei file raw su GitHub
TRANSCRIPT_URL = "https://raw.githubusercontent.com/wooters/berp-trans/master/transcript.txt"
WORDHIST_URL   = "https://raw.githubusercontent.com/wooters/berp-trans/master/wordhist.txt"

def download_file(url):
    """
    Scarica il contenuto di un file di testo da GitHub (o altra URL)
    e lo restituisce come lista di righe.
    """
    response = requests.get(url)
    response.raise_for_status()
    text = response.text
    lines = text.strip().split("\n")
    return lines

def tokenize(line):
    """
    Tokenizza la riga esattamente come il comando bash:
      - Divide la riga sul carattere spazio (' ')
      - Rimuove il primo campo
      - Non rimuove token vuoti (rispettando il comportamento di tr)
    """
    parts = line.rstrip("\n").split(" ")
    return parts[1:] if len(parts) > 1 else []

def build_unigram_bigram_counts(transcript_lines):
    """
    Data una lista di righe del transcript, costruisce:
      - un dizionario di conteggio degli unigrammi
      - un dizionario di conteggio dei bigrammi
    La tokenizzazione viene eseguita in modo "bash-like" e
    viene aggiunto <s> all'inizio e </s> alla fine di ogni frase.
    """
    unigram_counts = defaultdict(int)
    bigram_counts = defaultdict(int)
    
    for line in transcript_lines:
        # Tokenizza la riga e aggiunge i boundary
        tokens = tokenize(line)
        tokens = ["<s>"] + tokens + ["</s>"]
        
        for i, token in enumerate(tokens):
            unigram_counts[token] += 1
            if i < len(tokens) - 1:
                bigram_counts[(token, tokens[i + 1])] += 1
    
    return unigram_counts, bigram_counts

def compute_bigram_probabilities(unigram_counts, bigram_counts):
    """
    Calcola le probabilità P(w2|w1) = count(w1,w2)/count(w1)
    Restituisce un dizionario: bigram_probs[(w1, w2)] = P(w2|w1).
    """
    bigram_probs = {}
    for (w1, w2), count in bigram_counts.items():
        bigram_probs[(w1, w2)] = count / float(unigram_counts[w1])
    return bigram_probs

def phrase_probability(phrase_tokens, bigram_probs):
    """
    Calcola la probabilità di una frase (data come lista di token senza boundary)
    usando il modello di catena di Markov (bigrammi). Aggiunge i token
    di inizio (<s>) e fine frase (</s>) e calcola:
      P(phrase) = P(t1|<s>) * P(t2|t1) * ... * P(</s>|t_n)
      
    Restituisce un valore float.
    """
    # Aggiunge i token di inizio e fine frase
    tokens = ["<s>"] + phrase_tokens + ["</s>"]
    prob = 1.0
    for i in range(len(tokens) - 1):
        bigram = (tokens[i], tokens[i+1])
        p_bigram = bigram_probs.get(bigram, 0.0)
        prob *= p_bigram
    return prob

# Download dei file transcript e wordhist da GitHub
print("Download dei file transcript e wordhist da GitHub...")
transcript_lines = download_file(TRANSCRIPT_URL)
wordhist_lines   = download_file(WORDHIST_URL)

# Salvo i transcript in un nuovo file .txt
print("Salvo i transcript in un nuovo file .txt...")
with open("transcript.txt", 'w', encoding="utf-8") as f:
    f.write("\n".join(transcript_lines))

print("Costruzione dei conteggi di unigrammi e bigrammi dal transcript...")
unigram_counts, bigram_counts = build_unigram_bigram_counts(transcript_lines)

# Calcoliamo le probabilità di bigramma
bigram_probs = compute_bigram_probabilities(unigram_counts, bigram_counts)

# Esempi di frasi da valutare (senza boundary, che verranno aggiunti nella funzione)
phrase1 = ["i", "want", "english", "food"]
phrase2 = ["i", "want", "chinese", "food"]

p_phrase1 = phrase_probability(phrase1, bigram_probs)
p_phrase2 = phrase_probability(phrase2, bigram_probs)

print(f"Probabilità frase '{' '.join(phrase1)}': {p_phrase1:.8f}")
print(f"Probabilità frase '{' '.join(phrase2)}': {p_phrase2:.8f}")

# Esempio: Stampa top 5 unigrammi e top 5 bigrammi più frequenti
print("\nTop 7 Unigrammi (per conteggio):")
top5_uni = sorted(unigram_counts.items(), key=lambda x: x[1], reverse=True)[:7]
for w, c in top5_uni:
    print(f"{w}: {c}")

print("\nTop 5 Bigrammi (per conteggio):")
top5_bi = sorted(bigram_counts.items(), key=lambda x: x[1], reverse=True)[:5]
for (w1, w2), c in top5_bi:
    print(f"({w1}, {w2}): {c}")

Download dei file transcript e wordhist da GitHub...
Salvo i transcript in un nuovo file .txt...
Costruzione dei conteggi di unigrammi e bigrammi dal transcript...
Probabilità frase 'i want english food': 0.00000000
Probabilità frase 'i want chinese food': 0.00016241

Top 7 Unigrammi (per conteggio):
<s>: 8566
</s>: 8566
i: 2816
to: 2711
like: 1522
food: 1242
about: 1154

Top 5 Bigrammi (per conteggio):
(<s>, i): 1922
(like, to): 1172
(i, want): 908
(food, </s>): 806
(to, eat): 753


In [3]:
import pandas as pd

def generate_bigram_table(bigram_counts, words):
    """
    Genera una tabella (DataFrame) contenente i conteggi dei bigrammi per le parole specificate,
    includendo i token di inizio frase <s> e di fine frase </s>.
    
    Parametri:
      - bigram_counts: dizionario dei bigrammi (chiave = (w1, w2), valore = conteggio)
      - words: lista di parole (stringhe) da includere come righe e colonne
      
    Restituisce:
      - df: DataFrame con righe e colonne corrispondenti alle parole (comprese <s> e </s>),
            contenente i conteggi dei bigrammi.
    """
    # Assicuriamoci di includere i token di inizio e fine frase
    if "<s>" not in words:
        words = ["<s>"] + words
    if "</s>" not in words:
        words = words + ["</s>"]
        
    table = {}
    for w1 in words:
        row = {}
        for w2 in words:
            row[w2] = bigram_counts.get((w1, w2), 0)
        table[w1] = row
    df = pd.DataFrame.from_dict(table, orient='index')
    # Riordina le righe e le colonne in base all'ordine della lista "words"
    df = df.loc[words, words]
    return df

# Definisci l'insieme di parole da visualizzare nella tabella
words = ["i", "want", "to", "eat", "chinese", "food", "lunch", "spend"]

# Genera la tabella dei bigrammi utilizzando i conteggi ottenuti precedentemente (bigram_counts)
df_bigram = generate_bigram_table(bigram_counts, words)

# Stampa la tabella in formato Markdown
df_bigram.head(10)

,<s>,i,want,to,eat,chinese,food,lunch,spend,</s>
<s>,0,1922,4,32,4,10,4,39,1,0
i,0,1,908,0,12,0,0,0,2,0
want,0,2,0,673,0,7,6,6,1,2
to,0,0,0,2,753,3,0,6,233,3
eat,0,0,0,0,0,16,2,52,0,10
chinese,0,4,0,0,0,0,99,1,0,10
food,0,14,0,13,0,0,0,0,0,806
lunch,0,1,0,0,0,0,1,0,0,221
spend,0,0,0,1,0,0,0,0,0,8
</s>,0,0,0,0,0,0,0,0,0,0


In [4]:
for x in ["<s>", "i", "want", "to", "eat", "english", "food", "lunch", "spend", "</s>"]:
    print(x, unigram_counts[x])

<s> 8566
i 2816
want 1038
to 2711
eat 829
english 2
food 1242
lunch 392
spend 310
</s> 8566


In [5]:
df_bigram

,<s>,i,want,to,eat,chinese,food,lunch,spend,</s>
<s>,0,1922,4,32,4,10,4,39,1,0
i,0,1,908,0,12,0,0,0,2,0
want,0,2,0,673,0,7,6,6,1,2
to,0,0,0,2,753,3,0,6,233,3
eat,0,0,0,0,0,16,2,52,0,10
chinese,0,4,0,0,0,0,99,1,0,10
food,0,14,0,13,0,0,0,0,0,806
lunch,0,1,0,0,0,0,1,0,0,221
spend,0,0,0,1,0,0,0,0,0,8
</s>,0,0,0,0,0,0,0,0,0,0


In [6]:
len(unigram_counts)

1996

In [7]:
# Calcolo della matrice delle proabilità

N = df_bigram.apply(
        lambda row: row / unigram_counts[row.name],
        axis=1
    )
N.head(10)

,<s>,i,want,to,eat,chinese,food,lunch,spend,</s>
<s>,0.0,0.224375,0.000467,0.003736,0.000467,0.001167,0.000467,0.004553,0.000117,0.000000
i,0.0,0.000355,0.322443,0.000000,0.004261,0.000000,0.000000,0.000000,0.000710,0.000000
want,0.0,0.001927,0.000000,0.648362,0.000000,0.006744,0.005780,0.005780,0.000963,0.001927
to,0.0,0.000000,0.000000,0.000738,0.277757,0.001107,0.000000,0.002213,0.085946,0.001107
eat,0.0,0.000000,0.000000,0.000000,0.000000,0.019300,0.002413,0.062726,0.000000,0.012063
chinese,0.0,0.020725,0.000000,0.000000,0.000000,0.000000,0.512953,0.005181,0.000000,0.051813
food,0.0,0.011272,0.000000,0.010467,0.000000,0.000000,0.000000,0.000000,0.000000,0.648953
lunch,0.0,0.002551,0.000000,0.000000,0.000000,0.000000,0.002551,0.000000,0.000000,0.563776
spend,0.0,0.000000,0.000000,0.003226,0.000000,0.000000,0.000000,0.000000,0.000000,0.025806
</s>,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [11]:
# Applicazione del Laplace Smoothing

def apply_laplace_smoothing(df, unigram_counts):
    """
    Applica il Laplace smoothing (add-one smoothing) al DataFrame dei conteggi dei bigrammi.
    Aggiunge 1 ad ogni conteggio e normalizza ogni riga per ottenere delle probabilità condizionali.

    Parametri:
      - df: DataFrame con conteggi dei bigrammi (righe: parola corrente, colonne: parola successiva)
      - unigram_counts: dizionario con i conteggi dei unigrammi

    Restituisce:
      - df_smoothed: DataFrame con le probabilità condizionali smoothed.
      
    La formula applicata per ogni bigramma (w_{n-1}, w_n) è:
      P(w_n | w_{n-1}) = (c(w_{n-1}, w_n) + 1) / (c(w_{n-1}) + V)
    """
    # Calcola la dimensione del vocabolario
    V = len(unigram_counts)
    
    # Applica il Laplace smoothing per ogni riga del DataFrame
    df_smoothed = df.apply(
        lambda row:  (row + 1) / (unigram_counts[row.name] + V), 
        axis=1
    )
    
    return df_smoothed

# Definisci l'insieme di parole da visualizzare nella tabella
words = ["i", "want", "ot", "eat", "chinese", "food", "lunch", "spend"]

# Genera la tabella dei bigrammi utilizzando i conteggi ottenuti precedentemente (bigram_counts)
df_bigram = generate_bigram_table(bigram_counts, words)

# Applica il Laplace smoothing
df_bigram_smoothed = apply_laplace_smoothing(df_bigram, unigram_counts)

# Stampa la tabella in formato Markdown (le prime 8 righe)
df_bigram_smoothed.head(10)

,<s>,i,want,ot,eat,chinese,food,lunch,spend,</s>
<s>,0.000095,0.182051,0.000473,0.000095,0.000473,0.001041,0.000473,0.003787,0.000189,0.000095
i,0.000208,0.000416,0.188863,0.000208,0.002701,0.000208,0.000208,0.000208,0.000623,0.000208
want,0.000329,0.000988,0.000329,0.000329,0.000329,0.002636,0.002306,0.002306,0.000659,0.000988
ot,0.000501,0.000501,0.000501,0.000501,0.000501,0.000501,0.000501,0.000501,0.000501,0.000501
eat,0.000354,0.000354,0.000354,0.000354,0.000354,0.006016,0.001062,0.018754,0.000354,0.003892
chinese,0.000457,0.002283,0.000457,0.000457,0.000457,0.000457,0.045662,0.000913,0.000457,0.005023
food,0.000309,0.004631,0.000309,0.000309,0.000309,0.000309,0.000309,0.000309,0.000309,0.249151
lunch,0.000419,0.000837,0.000419,0.000419,0.000419,0.000419,0.000837,0.000419,0.000419,0.092926
spend,0.000433,0.000433,0.000433,0.000433,0.000433,0.000433,0.000433,0.000433,0.000433,0.003901
</s>,0.000095,0.000095,0.000095,0.000095,0.000095,0.000095,0.000095,0.000095,0.000095,0.000095


In [12]:
0.182051 * 0.188863 * 0.000329 * 0.000501 * 0.000329 * 0.000501 * 0.006016 * 0.045662 * 0.249151

6.393410535159305e-20

In [9]:
def sentence_probability(sentence, smoothed_df):
    """
    Calcola la probabilità di una frase come prodotto delle probabilità condizionali dei bigrammi.
    
    Parametri:
      - sentence: stringa contenente la frase.
      - smoothed_df: DataFrame contenente le probabilità condizionali (smoothed) dei bigrammi.
      
    Restituisce:
      - prob: probabilità della frase.
    """
    tokens = sentence.split()  # Tokenizzazione semplice basata sugli spazi
    prob = 1.0
    # Calcola il prodotto delle probabilità per ogni bigramma nella frase
    for i in range(len(tokens) - 1):
        w1 = tokens[i]
        w2 = tokens[i + 1]
        if w1 in smoothed_df.index and w2 in smoothed_df.columns:
            prob *= smoothed_df.loc[w1, w2]
        else:
            # Se una parola non è presente nel vocabolario, la probabilità è 0
            return 0.0
    return prob

# Esempio di frasi
sentence1 = phrase1 = " ".join(["i", "want", "english", "food"]) #"i want to eat chinese food"
sentence2 = phrase2 = " ".join(["i", "want", "chinese", "food"]) #"i want to spend lunch"

# Calcola le probabilità delle frasi usando il dataframe smoothed dei bigrammi
prob_sentence1 = sentence_probability(sentence1, df_bigram_smoothed)
prob_sentence2 = sentence_probability(sentence2, df_bigram_smoothed)

print("Probability of sentence1:", prob_sentence1)
print("Probability of sentence2:", prob_sentence2)

Probability of sentence1: 0.0
Probability of sentence2: 2.2754479913457698e-05
